# Day 09 · 模块与高级特性

> **前置**: Day01-08 全部(重点:函数/OOP/文件 I/O)
> **关键问题**: 代码破千行后,单文件不堪重负 —— 怎么把代码**拆分到多个文件**、让重复的"计时/日志/资源管理"逻辑自动复用?本节是工具箱补完课:包、生成器、上下文管理器、装饰器

**引入(5 分钟)**

**第 1 讲:import 与包**

三种导入方式:`import 模块`(带前缀访问)、`from 模块 import 成员`(直接用)、`from 模块 import 成员 as 别名`(改名)。把函数/类放进 `utils.py`,同目录 `import utils` 即可使用。**包**(Package) = 模块的文件夹,必须含 `__init__.py`(可以为空),让 Python 把文件夹识别为包。口诀:**`import 模块` 带前缀,`from` 直接用,`as` 改别名**。

In [ ]:
# ===== 三种 import 方式 =====

# 方式 1:import 模块 —— 通过模块名访问成员
import math
print(math.sqrt(16))            # 4.0,通过模块名.成员名访问

# 方式 2:from 模块 import 成员 —— 直接用成员名,不再写模块名前缀
from math import sqrt
print(sqrt(16))                 # 4.0,直接用函数名

# 方式 3:from 模块 import 成员 as 别名 —— 用别名替代原函数名
from math import factorial as fac
print(fac(5))                   # 120,用别名 fac 调用

In [ ]:
# ===== 包目录结构示例(伪代码说明) =====
# 实际项目中需要创建如下文件结构:
#
# mypkg/
# ├── __init__.py      <- 必须有,可为空文件,让 Python 识别为包
# ├── tools.py         <- 放工具函数,例如 greet(name)
# └── helpers.py       <- 放辅助函数,例如 add(a, b)
#
# 主程序(main.py)中这样使用:
# from mypkg.tools import greet
# from mypkg.helpers import add
# greet()
# add(1, 2)

# __init__.py 也可以在里面写初始化逻辑或 __all__ 控制导出
# 例如在 __init__.py 中写: from .tools import greet

---

**第 2 讲:生成器 yield**

普通函数 `return` 一次就结束,想**逐个产出**数据且**不占内存**,用 `yield`。生成器是**惰性计算**的迭代器。口诀:**`yield` 是 `return` 的"Hold"版本 —— 产出值但不退场**,下次 `next()` 从这里恢复。

生成器 vs 列表:列表全量存内存,占空间但可多次遍历;生成器产一个放一个,省内存但只能遍历一次。

In [ ]:
def 平方(limit):
    # 生成器函数:用 yield 逐个产出,而不是一次性返回列表
    for i in range(limit):
        yield i * i             # 每次产一个值,暂停,下次从这里恢复

gen = 平方(4)                    # 调用生成器函数返回生成器对象
print(next(gen))                # 0 —— 第一次取,产出 0*0
print(next(gen))                # 1 —— 暂停处恢复,产出 1*1
print(next(gen))                # 4 —— 暂停处恢复,产出 2*2

# 生成器也支持 for 循环遍历
for x in 平方(3):
    print("for:", x)            # for: 0  for: 1  for: 4

# 重要:生成器只能用一次,再次遍历是空的
gen2 = 平方(3)
print(list(gen2))               # [0, 1, 4]
print(list(gen2))               # [] —— 第二次遍历空了

---

**第 3 讲:上下文管理器(__enter__ / __exit__)**

`with` 语句需要对象实现 `__enter__`(进时自动执行)和 `__exit__`(出时自动执行)两个方法。`__exit__` 必须接受三个 `exc_` 参数(异常类型/值/ traceback),即使不用也要写。类比 Day07 的 `with open` 自动关文件 —— 这个自动关的动作就是 `__exit__` 在做事。

In [ ]:
class Timer:
    # 自定义上下文管理器:自动计时

    def __enter__(self):
        # 进入 with 块时自动调用,记录开始时间
        import time
        self.start = time.time()
        return self            # 返回值赋给 as 后的变量

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 退出 with 块时自动调用,打印耗时
        # exc_type/exc_val/exc_tb 是异常信息,无异常时都是 None
        import time
        print(f"耗时 {time.time() - self.start:.4f}s")

# with 块退出时自动调 __exit__ 计时
with Timer():
    sum(range(1000000))     # 自动计时这段代码的耗时

---

**第 4 讲:装饰器(@timer)**

装饰器本质是**接收函数 -> 返回新函数**的高阶函数,语法糖 `@decorator`。口诀:**装饰器 = 把函数装进壳,`@` 语法糖,`functools.wraps` 留名**。**永远写 `@functools.wraps(fn)`**,否则被装饰的函数名会变成 `wrapper`,调试/文档全部出错。需要参数时再加一层:外层接收参数,中层接收函数,内层是真正的包装。

In [ ]:
import functools, time

def timer(fn):
    # 装饰器:给函数加计时功能,不修改原函数代码
    @functools.wraps(fn)        # 保留原函数名和文档字符串
    def wrapper(*args, **kwargs):
        # wrapper 是真正的包装函数,代替原函数被调用
        start = time.time()     # 记录开始时间
        result = fn(*args, **kwargs)  # 调用原函数
        # 调用结束后打印耗时
        print(f"{fn.__name__} 耗时 {time.time() - start:.4f}s")
        return result           # 返回原函数的结果
    return wrapper              # 返回包装后的函数

@timer                          # 等价于 slow = timer(slow)
def slow():
    time.sleep(0.1)             # 模拟耗时操作

slow()                          # slow 耗时 0.1xxx s
print(slow.__name__)            # slow(被 @functools.wraps 保留,不是 wrapper)

In [ ]:
import functools

def log(level="INFO"):
    # 带参数的装饰器:外层接收参数 level
    def decorator(fn):
        # 中层接收函数 fn
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # 内层是真正的包装逻辑
            print(f"[{level}] 调用 {fn.__name__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@log("DEBUG")                   # 等价于 add = log("DEBUG")(add)
def add(a, b):
    return a + b

print(add(1, 2))                # [DEBUG] 调用 add -> 3

---

**今日小结**

`import` / `from` / `as` 三件套;`yield` 生成器惰性产出;`__enter__` / `__exit__` 自定义上下文管理器;`@decorator` + `@functools.wraps` 装饰器。

易错点:自定义装饰器漏写 `@functools.wraps`,原函数名丢失变成 `wrapper`;把生成器当列表用 —— 取 `len()` 报错,或遍历两次第二轮空;上下文管理器 `__exit__` 不加三个 `exc_` 参数,签名报错;包目录必须有 `__init__.py`(Python 3.3+ 虽非强制,但最好保留),否则 IDE 不认识。

**更多练习**

- 当堂练:course/lesson09/in_class/practice01-06.py
- 课后作业:course/lesson09/homework/task01-03.py